# 02 — Data Preprocessing

**Objective:** Clean the raw Online Retail dataset so it is ready for Exploratory Data Analysis and Market Basket Analysis (Apriori / FP-Growth).

### Preprocessing Steps
1. Load the dataset
2. Handle missing values
3. Remove duplicate records
4. Handle cancelled transactions
5. Handle returns and invalid quantities
6. Convert data types
7. Save the cleaned dataset

A **preprocessing summary** is maintained throughout to track the impact of each step.

---
## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('../dataset/raw/Online_Retail_enlargedcsv.csv', encoding='latin1')

initial_rows = len(df)
initial_cols = df.shape[1]

print(f"Dataset loaded successfully.")
print(f"Initial shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

### Initialize Preprocessing Summary Tracker

In [ ]:
# Track rows removed at each step
summary_log = []

def log_step(step_name, rows_before, rows_after):
    """Log a preprocessing step with before/after row counts."""
    removed = rows_before - rows_after
    pct = round(removed / rows_before * 100, 2) if rows_before > 0 else 0
    summary_log.append({
        'Step': step_name,
        'Rows Before': rows_before,
        'Rows Removed': removed,
        'Rows After': rows_after,
        '% Removed': pct
    })
    print(f"[{step_name}]  {rows_before} → {rows_after}  |  Removed: {removed} ({pct}%)")

print("Summary tracker initialized.")

---
## 3. Handle Missing Values

In [ ]:
# Check missing values before cleaning
print("Missing values per column:")
print("-" * 40)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
pd.DataFrame({'Missing': missing, '% Missing': missing_pct})

### 3a. Remove rows with missing CustomerID

Market basket analysis requires a valid customer or transaction identifier to group items into baskets. Records without `CustomerID` cannot be assigned to any transaction basket, so they must be dropped.

In [ ]:
rows_before = len(df)
df = df.dropna(subset=['CustomerID'])
log_step('Remove missing CustomerID', rows_before, len(df))

### 3b. Remove rows with missing Description

Rows without a product description cannot be meaningfully interpreted in association rules.

In [ ]:
rows_before = len(df)
df = df.dropna(subset=['Description'])
log_step('Remove missing Description', rows_before, len(df))

In [ ]:
# Verify no missing values remain
print("Remaining missing values:")
print(df.isnull().sum())

---
## 4. Remove Duplicate Records

In [ ]:
print(f"Duplicate rows found: {df.duplicated().sum()}")

In [ ]:
rows_before = len(df)
df = df.drop_duplicates()
log_step('Remove duplicate records', rows_before, len(df))

---
## 5. Handle Cancelled Transactions

Cancelled orders have an `InvoiceNo` starting with **'C'** (e.g., C536379). These are not valid purchase transactions and must be excluded from market basket analysis.

In [ ]:
# Identify cancelled transactions
cancelled_mask = df['InvoiceNo'].astype(str).str.startswith('C')
print(f"Cancelled transactions found: {cancelled_mask.sum()}")
print(f"Percentage of data: {round(cancelled_mask.sum() / len(df) * 100, 2)}%")

In [ ]:
# Sample cancelled transactions
df[cancelled_mask].head()

In [ ]:
rows_before = len(df)
df = df[~cancelled_mask]
log_step('Remove cancelled transactions (C-prefix)', rows_before, len(df))

---
## 6. Handle Returns and Invalid Quantities

Even after removing C-prefix invoices, some rows may still have:
- **Quantity ≤ 0** — returns, adjustments, or data-entry errors
- **UnitPrice ≤ 0** — free items, test entries, or errors

These are not valid purchase transactions for market basket analysis.

In [ ]:
# Check for invalid quantities
print(f"Rows with Quantity <= 0 : {(df['Quantity'] <= 0).sum()}")
print(f"Rows with UnitPrice <= 0: {(df['UnitPrice'] <= 0).sum()}")

In [ ]:
# Remove rows where Quantity <= 0
rows_before = len(df)
df = df[df['Quantity'] > 0]
log_step('Remove Quantity <= 0', rows_before, len(df))

In [ ]:
# Remove rows where UnitPrice <= 0
rows_before = len(df)
df = df[df['UnitPrice'] > 0]
log_step('Remove UnitPrice <= 0', rows_before, len(df))

---
## 7. Convert Data Types

In [ ]:
# Check current data types
print("Data types BEFORE conversion:")
print(df.dtypes)

In [ ]:
# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Convert CustomerID from float to integer (NaNs already removed)
df['CustomerID'] = df['CustomerID'].astype(int)

# Convert InvoiceNo to string (ensure consistency)
df['InvoiceNo'] = df['InvoiceNo'].astype(str)

# Convert StockCode to string
df['StockCode'] = df['StockCode'].astype(str)

# Strip whitespace from Description
df['Description'] = df['Description'].str.strip()

print("Data types AFTER conversion:")
print(df.dtypes)

### 7a. Reset Index

In [ ]:
df = df.reset_index(drop=True)
print(f"Index reset. Final shape: {df.shape}")

---
## 8. Final Verification

In [ ]:
print("=" * 50)
print("       FINAL DATA QUALITY CHECK")
print("=" * 50)
print(f"Rows           : {len(df)}")
print(f"Columns        : {df.shape[1]}")
print(f"Missing values : {df.isnull().sum().sum()}")
print(f"Duplicates     : {df.duplicated().sum()}")
print(f"Cancelled (C)  : {df['InvoiceNo'].str.startswith('C').sum()}")
print(f"Quantity <= 0  : {(df['Quantity'] <= 0).sum()}")
print(f"UnitPrice <= 0 : {(df['UnitPrice'] <= 0).sum()}")
print("=" * 50)

In [ ]:
# Quick look at cleaned data
df.head(10)

In [ ]:
df.info()

---
## 9. Preprocessing Summary

In [ ]:
summary_df = pd.DataFrame(summary_log)
summary_df

In [ ]:
# Overall summary
total_removed = initial_rows - len(df)
total_pct = round(total_removed / initial_rows * 100, 2)

print("=" * 55)
print("          PREPROCESSING SUMMARY")
print("=" * 55)
print(f"  Initial records       : {initial_rows:>10,}")
print(f"  Final records         : {len(df):>10,}")
print(f"  Total rows removed    : {total_removed:>10,}  ({total_pct}%)")
print(f"  Records retained      : {round(100 - total_pct, 2)}%")
print("=" * 55)
print()
print("Step-wise breakdown:")
print("-" * 55)
for _, row in summary_df.iterrows():
    print(f"  {row['Step']:<42} : -{row['Rows Removed']:>7,} ({row['% Removed']}%)")

---
## 10. Save the Cleaned Dataset

In [ ]:
# Save cleaned data
output_path = '../dataset/processed/cleaned_data.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")

In [ ]:
# Save preprocessing summary
summary_df.to_csv('../results/preprocessing_summary.csv', index=False)
print("Preprocessing summary saved to: ../results/preprocessing_summary.csv")

---
*Next step → `03_eda.ipynb` — Exploratory Data Analysis on the cleaned dataset.*